In [0]:
from pyspark.sql import functions as f,Window
from pyspark.sql.functions import regexp_replace
from pyspark.sql.types import *

data = [
    # Valid Record
    (101, ' Rahul ', 'RAHUL@GMAIL.COM ', ' BANGALORE ', 50000, '98765-43210', '2025-01-02'),
    # Older version of customer 101
    (101, ' Rahul ', 'rahul@gmail.com', 'Bangalore', 50000, '98765-43210', '2025-01-01'),
    # Exact duplicate
    (101, ' Rahul ', 'rahul@gmail.com', 'Bangalore', 50000, '98765-43210', '2025-01-01'),
    # Null email
    (102, 'Amit', None, 'Pune', 60000, '98765-55210', '2025-01-05'),
    # Invalid email
    (103, 'John', 'johngmail.com', 'Chennai', 70000, '98765-00210', '2025-01-04'),
    # Null name
    (104, None, 'david@gmail.com', 'Mumbai', 55000, '98765-00910', '2025-01-03'),
    # Latest record for customer 104
    (104, ' DAVID ', 'DAVID@GMAIL.COM ', ' MUMBAI ', 56000, '98765-00910', '2025-01-10'),
    # Exact duplicate
    (104, ' DAVID ', 'DAVID@GMAIL.COM ', ' MUMBAI ', 56000, '98765-00910', '2025-01-10'),
    # Null city
    (105, 'Sara', 'sara@gmail.com', None, 80000, '99865-88810', '2025-01-14'),
    # Null salary
    (106, 'Ravi', 'ravi@gmail.com', 'Hyderabad', None, '98765-11110', '2025-01-12'),
    # Negative salary
    (107, 'Kiran', 'kiran@gmail.com', 'Delhi', -25000, '98765-22210', '2025-01-07'),
    # Invalid phone (letters)
    (108, 'Ajay', 'ajay@gmail.com', 'Noida', 45000, '98AB5-22210', '2025-01-09'),
    # Invalid phone (9 digits after cleanup)
    (109, 'Neha', 'neha@gmail.com', 'Pune', 55000, '9876-54321', '2025-01-08'),
    # Future load date
    (110, 'Manoj', 'manoj@gmail.com', 'Chennai', 65000, '98765-33310', '2030-01-01'),
    # Null customer_id
    (None, 'Ramesh', 'ramesh@gmail.com', 'Mysore', 45000, '98765-44410', '2025-01-05'),
    # Multiple nulls
    (111, None, None, None, None, None, '2025-01-11'),
    # Invalid email (double @)
    (112, 'Anu', 'anu@@gmail.com', 'Kolkata', 50000, '98765-55510', '2025-01-06'),
    # Email with spaces
    (113, ' Priya ', ' PRIYA@GMAIL.COM ', ' Chennai ', 72000, '98765-66610', '2025-01-13'),
    # Phone with spaces
    (114, 'Vijay', 'vijay@gmail.com', 'Bangalore', 90000, '98765 77710', '2025-01-10'),
    # Another valid record
    (115, 'Arjun', 'arjun@gmail.com', 'Pune', 45000, '98765-88811', '2025-01-05')
]

schema = StructType([
    StructField("customer_id", IntegerType()),
    StructField("customer_name", StringType()),
    StructField("customer_email", StringType()),
    StructField("customer_city", StringType()),
    StructField("customer_salary", IntegerType()),
    StructField("customer_phone", StringType()),
    StructField("load_dt", StringType())
])

customers = spark.createDataFrame(data=data,schema=schema)
customers = customers.dropna(subset=('customer_id'))
customers = customers.withColumn('load_dt',f.col('load_dt').cast(DateType()))
customers = customers.dropDuplicates()
w = Window.partitionBy('customer_id').orderBy(f.col('load_dt').desc())
customers  = customers.withColumn('rn',f.row_number().over(w)).filter(f.col('rn')==1)
customers = customers.fillna(
   {'customer_name':'Unknown',"customer_email":'Not Available','customer_city':'Unknown City','customer_salary':0}
)

customers = customers.withColumn('customer_name',f.initcap(f.trim(f.col('customer_name')))).withColumn('customer_email',f.lower(f.trim(f.col('customer_email')))).withColumn('customer_city',f.initcap(f.trim(f.col('customer_city')))).withColumn('customer_phone',regexp_replace('customer_phone', '-', ''))

# Data Validation
customers_vaild = customers.filter(
    (f.col('customer_email').like("%_@__%.__%")) &
    (f.length(f.col('customer_phone')) == 10) &
    (f.col('customer_salary') > 0)
).select('customer_id','customer_name','customer_email','customer_city','customer_salary','customer_phone','load_dt')

customers_vaild = customers_vaild.withColumn('data_quality_status',f.lit('Pass'))

ids = customers_vaild.select(f.collect_list('customer_id')).first()[0]

customers_invaild = customers.where(~f.col('customer_id').isin(ids)).select('customer_id','customer_name','customer_email','customer_city','customer_salary','customer_phone','load_dt').withColumn('data_quality_status',f.lit('Fail'))

customers_vaild.show()
customers_invaild.show()

customers_invaild.write.format('delta').mode('overwrite').option('overwriteSchema','true').saveAsTable('error_customers')
customers_vaild.write.format('delta').mode('overwrite').option('overwriteSchema','true').saveAsTable('silver_customers')
